In [ ]:
import pandas as pd
from rail_connectors.connection import connect

excel_path = "/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx"
month_start = "2026-03-01"
month_end = "2026-03-31"

def norm_agr(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def norm_inn(s: pd.Series) -> pd.Series:
    x = (
        s.astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
        .str.replace(r"\\D", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

    # Если в Excel потерялся ведущий 0, восстанавливаем длину до 10/12.
    def _fix_len(v):
        if pd.isna(v):
            return pd.NA
        n = len(v)
        if n == 9:
            return v.zfill(10)
        if n == 11:
            return v.zfill(12)
        if n in (10, 12):
            return v
        return pd.NA

    return x.map(_fix_len)

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()


In [ ]:
# Секция 1: 10 ИНН из Excel (есть agr_id), но нет agr_id в ods_alpha.scd1_agreements за март
excel = pd.read_excel(excel_path).copy()
excel['agr_id_norm'] = norm_agr(excel['ID договора'])
excel['inn_norm'] = norm_inn(excel['ИНН'])

with imp:
    agr = imp.fetch(f"""
        select distinct cast(abs_agr_id as string) as agr_id
        from ods_alpha.scd1_agreements
        where upper(trim(cast(acq_class as string))) = 'SA'
          and abs_agr_id is not null
          and cast(d_valid_from as date) <= cast('{month_end}' as date)
          and (d_valid_to is null or cast(d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(ods_deleted_flg, '0') <> '1'
    """)

agr_set = set(norm_agr(agr['agr_id']).dropna().unique().tolist()) if agr is not None else set()

inn_list_1 = (
    excel[
        excel['agr_id_norm'].notna() &
        (~excel['agr_id_norm'].isin(agr_set)) &
        excel['inn_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .sort_values()
    .head(10)
    .tolist()
)

for i, inn in enumerate(inn_list_1, 1):
    print(f"{i}. {inn}")


In [ ]:
# Секция 2: 10 ИНН по agr_id из r2_ip_merchants (март по датам), которых нет в agreements за март
with imp:
    r2 = imp.fetch(f"""
        select distinct
            cast(m.id as string) as agr_id,
            regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_client cl
          on cast(cl.id as string) = cast(m.c_cl_org as string)
        where cast(m.c_date_begin as date) <= cast('{month_end}' as date)
          and (
                m.c_date_close is null
             or cast(m.c_date_close as date) >= cast('{month_start}' as date)
          )
    """)

if r2 is None:
    r2 = pd.DataFrame(columns=['agr_id', 'inn'])

r2['agr_id_norm'] = norm_agr(r2['agr_id'])
r2['inn_norm'] = norm_inn(r2['inn'])

inn_list_2 = (
    r2[
        r2['agr_id_norm'].notna() &
        (~r2['agr_id_norm'].isin(agr_set)) &
        r2['inn_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .sort_values()
    .head(10)
    .tolist()
)

for i, inn in enumerate(inn_list_2, 1):
    print(f"{i}. {inn}")


In [ ]:
# Секция 3: по всем уникальным ИНН из Excel — есть ли для них agr_id в ods_alpha.scd1_agreements
excel_inn_all = excel['inn_norm'].dropna().drop_duplicates().sort_values().tolist()
excel_inn_sql = ", ".join([f"'{x}'" for x in excel_inn_all]) if excel_inn_all else "''"

with imp:
    inn_agr = imp.fetch(f"""
        select
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
            cast(a.abs_agr_id as string) as agr_id
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({excel_inn_sql})
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
    """)

if inn_agr is None:
    inn_agr = pd.DataFrame(columns=['inn', 'agr_id'])

inn_agr['inn_norm'] = norm_inn(inn_agr['inn'])
inn_agr['agr_id_norm'] = norm_agr(inn_agr['agr_id'])

inn_with_agr = set(
    inn_agr[
        inn_agr['inn_norm'].notna() &
        inn_agr['agr_id_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .tolist()
)

inn_without_agr = [x for x in excel_inn_all if x not in inn_with_agr]

print(f"Всего уникальных ИНН в Excel: {len(excel_inn_all)}")
print(f"Из них с найденным agr_id в agreements: {len(inn_with_agr)}")
print(f"Без найденного agr_id в agreements: {len(inn_without_agr)}")

for i, inn in enumerate(inn_without_agr, 1):
    print(f"{i}. {inn}")

In [ ]:
# Секция 4: май, SA в ods_alpha.scd1_agreements — количество уникальных ИНН, заполняемость agr_id, примеры 10 ИНН
may_start = "2026-05-01"
may_end = "2026-05-31"

with imp:
    may_sa = imp.fetch(f"""
        select
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
            cast(a.abs_agr_id as string) as agr_id
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{may_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{may_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
    """)

if may_sa is None:
    may_sa = pd.DataFrame(columns=['inn', 'agr_id'])

may_sa['inn_norm'] = norm_inn(may_sa['inn'])
may_sa['agr_id_norm'] = norm_agr(may_sa['agr_id'])

unique_inn_all = sorted(may_sa['inn_norm'].dropna().drop_duplicates().tolist())
unique_inn_with_agr = set(
    may_sa[
        may_sa['inn_norm'].notna() &
        may_sa['agr_id_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .tolist()
)

inn_total = len(unique_inn_all)
inn_with_agr = len(unique_inn_with_agr)
inn_fill_rate = round((inn_with_agr / inn_total) * 100, 2) if inn_total else 0.0

print(f"Уникальных ИНН (SA, май): {inn_total}")
print(f"Заполняемость agr_id по ИНН (есть хотя бы один agr_id): {inn_with_agr}/{inn_total} ({inn_fill_rate}%)")
print("Примеры 10 ИНН:")
for i, inn in enumerate(unique_inn_all[:10], 1):
    print(f"{i}. {inn}")

In [ ]:
# Секция 5: март, SA в ods_alpha.scd1_agreements — количество уникальных ИНН, заполняемость agr_id, примеры 10 ИНН
march_start = "2026-03-01"
march_end = "2026-03-31"

with imp:
    march_sa = imp.fetch(f"""
        select
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
            cast(a.abs_agr_id as string) as agr_id
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{march_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{march_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
    """)

if march_sa is None:
    march_sa = pd.DataFrame(columns=['inn', 'agr_id'])

march_sa['inn_norm'] = norm_inn(march_sa['inn'])
march_sa['agr_id_norm'] = norm_agr(march_sa['agr_id'])

unique_inn_all_march = sorted(march_sa['inn_norm'].dropna().drop_duplicates().tolist())
unique_inn_with_agr_march = set(
    march_sa[
        march_sa['inn_norm'].notna() &
        march_sa['agr_id_norm'].notna()
    ]['inn_norm']
    .drop_duplicates()
    .tolist()
)

inn_total_march = len(unique_inn_all_march)
inn_with_agr_march = len(unique_inn_with_agr_march)
inn_fill_rate_march = round((inn_with_agr_march / inn_total_march) * 100, 2) if inn_total_march else 0.0

print(f"Уникальных ИНН (SA, март): {inn_total_march}")
print(f"Заполняемость agr_id по ИНН (есть хотя бы один agr_id): {inn_with_agr_march}/{inn_total_march} ({inn_fill_rate_march}%)")
print("Примеры 10 ИНН:")
for i, inn in enumerate(unique_inn_all_march[:10], 1):
    print(f"{i}. {inn}")

In [ ]:
# Секция 6: ИНН за март есть в озере (SA-фильтр), но нет в Excel
excel_inn_set = set(excel['inn_norm'].dropna().drop_duplicates().tolist())

with imp:
    lake_march_inn = imp.fetch(f"""
        select distinct
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
    """)

if lake_march_inn is None:
    lake_march_inn = pd.DataFrame(columns=['inn'])

lake_march_inn['inn_norm'] = norm_inn(lake_march_inn['inn'])
lake_inn_set = set(lake_march_inn['inn_norm'].dropna().drop_duplicates().tolist())

inn_in_lake_not_in_excel = sorted(list(lake_inn_set - excel_inn_set))

print(f"ИНН в озере (март, SA): {len(lake_inn_set)}")
print(f"ИНН в Excel (март): {len(excel_inn_set)}")
print(f"ИНН есть в озере, но нет в Excel: {len(inn_in_lake_not_in_excel)}")

for i, inn in enumerate(inn_in_lake_not_in_excel, 1):
    print(f"{i}. {inn}")